# Bayesian networks - advanced walkthrough

Run every cell from top to bottom. The notebook prints intermediate values and draws visualizations so the math stays visible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)
plt.style.use('default')

## 1. Define the classic sprinkler network
Cloudy affects sprinkler and rain. Sprinkler and rain affect wet grass.

In [ ]:
P_cloudy = {True: 0.5, False: 0.5}
P_sprinkler = {True: {True: 0.1, False: 0.9}, False: {True: 0.5, False: 0.5}}
P_rain = {True: {True: 0.8, False: 0.2}, False: {True: 0.2, False: 0.8}}
P_wet = {
    (True, True): {True: 0.99, False: 0.01},
    (True, False): {True: 0.90, False: 0.10},
    (False, True): {True: 0.90, False: 0.10},
    (False, False): {True: 0.00, False: 1.00},
}
print('Network: Cloudy -> Sprinkler, Cloudy -> Rain, Sprinkler/Rain -> WetGrass')

## 2. Enumerate the full joint distribution

In [ ]:
rows = []
for cloudy in [True, False]:
    for sprinkler in [True, False]:
        for rain in [True, False]:
            for wet in [True, False]:
                prob = (P_cloudy[cloudy] *
                        P_sprinkler[cloudy][sprinkler] *
                        P_rain[cloudy][rain] *
                        P_wet[(sprinkler, rain)][wet])
                rows.append({'Cloudy': cloudy, 'Sprinkler': sprinkler, 'Rain': rain, 'WetGrass': wet, 'probability': prob})
joint = pd.DataFrame(rows)
display(joint.head(10))
print('Sanity check, joint sums to:', joint['probability'].sum())

## 3. Infer P(Rain | WetGrass=true)

In [ ]:
evidence = joint[joint['WetGrass'] == True]
posterior = evidence.groupby('Rain')['probability'].sum() / evidence['probability'].sum()
posterior = posterior.rename('P(Rain | WetGrass=true)').reset_index()
display(posterior)
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(posterior['Rain'].astype(str), posterior['P(Rain | WetGrass=true)'], color=['#9ecae1', '#3182bd'])
ax.set_ylim(0, 1)
ax.set_title('Posterior after observing wet grass')
ax.set_xlabel('Rain')
ax.set_ylabel('probability')
plt.show()

## 4. Explaining away
If wet grass is observed, rain becomes less likely once sprinkler is also known to be on.

In [ ]:
wet = joint[joint['WetGrass'] == True]
wet_and_sprinkler = joint[(joint['WetGrass'] == True) & (joint['Sprinkler'] == True)]
p_rain_given_wet = wet[wet['Rain'] == True]['probability'].sum() / wet['probability'].sum()
p_rain_given_wet_sprinkler = wet_and_sprinkler[wet_and_sprinkler['Rain'] == True]['probability'].sum() / wet_and_sprinkler['probability'].sum()
compare = pd.DataFrame({
    'query': ['P(Rain | WetGrass)', 'P(Rain | WetGrass, Sprinkler)'],
    'probability': [p_rain_given_wet, p_rain_given_wet_sprinkler]
})
display(compare)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(compare['query'], compare['probability'], color=['#3182bd', '#de2d26'])
ax.set_ylim(0, 1)
ax.set_title('Explaining away in a Bayesian network')
ax.tick_params(axis='x', rotation=15)
plt.show()

Try changing `P_sprinkler[False][True]` from 0.5 to 0.1 and rerun the posterior cells.